# Fitness Learning using Layer-Aware Representations    

This model implements a Dynamic Gating Mechanism through Per-Layer Embeddings (PLE), a specialized architecture designed to maintain signal integrity in deep networks. 

1. The Contextual Key (Summation & Global Integration)  
The process begins by retrieving a unique "Identity Signal" for every layer from a massive embedding table. This is combined with a "Context Signal" (a projection of the current residual stream) to create a complete representation of a token’s meaning at a specific depth in the model.
* Identity Signal: "I am the word 'Apple'." (A constant reminder of the input token).   
* Context Signal: "The current sentence is about technology, not fruit." (The global awareness).    
* The Layer-Specific Key: "I am the word 'Apple' in a technology context at Layer 12."  

By adding these before they enter the block, we provide each layer with a pre-conditioned "key" that is already aware of the global intent. This is scaled by $2^{-0.5}$ to maintain numerical stability.   

2. The Soft-Switch (Multiplicative Gating)  
Once inside the block, the model uses the PLE signal to perform feature selection. While standard Transformers rely on addition (feature accumulation), the PLE mechanism uses multiplication to act as a soft-switch or filter.
* The Hidden States: These represent the complex transformations (grammar, syntax, logic) the model has calculated so far.
* The PLE Signal: Acts as a semantic mask or stencil.
* The Result: By multiplying the activated hidden states by the PLE signal, the model decides which specific features are relevant for this specific layer's task.

3. Summary of the Flow

| Stage | Operation | Purpose |
|---|---|---|
| Summation (Global) | Identity + Context | Merges who the token is with what the sentence is about. |
| Activation (Local) | $\text{SiLU}(\text{Linear}(x))$ | Prepares the current layer's features to be filtered. |
| Multiplication (Interaction) | $\text{Gate} \times \text{PLE}$ | Uses the "Who + What" signal to prune and refine the hidden stream. |



**Why this is Powerful**:   
Preventing Information DecayIn standard deep Transformers, the original input token often becomes "washed out" or blurry by the time it reaches the final layers. This architecture prevents Information Decay. Because the identity signal is injected directly from the input tokens into every single layer, the model never "forgets" the original word, even in 100+ layer architectures. This ensures that the final prediction is grounded in the original input, regardless of how much abstract reasoning happens in the middle layers.

# Tokenizer

In [4]:
import sys
sys.path.append('../')

from src.dataset.tokenizer import ProteinTokenizer

In [2]:
tokenizer = ProteinTokenizer()

encoded = tokenizer("MKTLLLTLVVVTIVCLDLGAVPNEEFSLEKKKX")

print(encoded["input_ids"])

[0, 21, 16, 12, 5, 5, 5, 12, 5, 8, 8, 8, 12, 13, 8, 24, 5, 14, 5, 7, 6, 8, 15, 18, 10, 10, 19, 9, 5, 10, 16, 16, 16, 25, 2]


In [11]:
tokenizer(["MKTLTL", "NE<mask>E"], padding=True, return_tensors='pt') 

{'input_ids': tensor([[ 0, 21, 16, 12,  5, 12,  5,  2],
        [ 0, 18, 10,  4, 10,  2,  1,  1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0]])}

In [18]:
decoded = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)
print(decoded)

MKTLLLTLVVVTIVCLDLGAVPNEEFSLEKKKX


In [14]:
tokenizer.special_token_ids

[2, 3, 1, 0, 4]

In [17]:
tokenizer.all_special_tokens

['<eos>', '<unk>', '<pad>', '<cls>', '<mask>']

In [3]:
print(tokenizer.pad_token_id,)
print(tokenizer.mask_token_id,)
print([tokenizer.cls_token_id, tokenizer.eos_token_id])
print(tokenizer.vocab_size)

1
4
[0, 2]
30


In [19]:
tokenizer.cls_token_id        # HuggingFace transformers
tokenizer.token_to_id("[CLS]") # HuggingFace tokenizers (fast)
tokenizer.cls_id              # ESM tokenizer (your case, likely)

AttributeError: ProteinTokenizer has no attribute token_to_id

In [23]:
len(tokenizer.all_token_ids)

30

In [ ]:
padding_token=tokenizer.pad_token_type_id,
mask_token=tokenizer.mask_token_id,
no_mask_tokens=[tokenizer.cls_token_id, tokenizer.eos_token_id ],
n_tokens=len(tokenizer.all_token_ids),

# ESM C architecture

In [ ]:
ESMC(
  (embed): Embedding(64, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-29): 30 x UnifiedTransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
          (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (rotary): RotaryEmbedding()
        )
        (ffn): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=5120, bias=False)
          (2): SwiGLU()
          (3): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (sequence_head): Sequential(
    (0): Linear(in_features=960, out_features=960, bias=True)
    (1): GELU(approximate='none')
    (2): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
    (3): Linear(in_features=960, out_features=64, bias=True)
  )
)

In [ ]:
BERT(
  (embed): Embedding(30, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-4): 5 x TransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (q_ln): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (RoPE): RotaryPositionalEncoding()
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (ffn): FFN(
          (ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (up_proj): Linear(in_features=960, out_features=5120, bias=False)
          (swiglu): SwiGLU()
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): LMHead(
    (proj): Linear(in_features=960, out_features=960, bias=True)
    (act): GELU(approximate='none')
    (ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
    (proj_final): Linear(in_features=960, out_features=30, bias=True)
  )
)

In [ ]:
FLARE(
  (embed): Embedding(30, 960)
  (ple_embed): PerLayerEmbeddings(
    (ple_embed_lookup_table): Embedding(30, 128)
    (ple_embed_context): Linear(in_features=960, out_features=128, bias=False)
    (ple_embed_context_norm): RMSNorm()
  )
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0): TransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (q_ln): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (RoPE): RotaryPositionalEncoding()
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (ffn): FFN(
          (ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (up_proj): Linear(in_features=960, out_features=5120, bias=False)
          (swiglu): SwiGLU()
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
        )
        (ple_module): PLEModulation(
          (proj_down): Linear(in_features=960, out_features=128, bias=False)
          (act_fn): SiLU()
          (proj_up): Linear(in_features=128, out_features=960, bias=False)
          (norm): RMSNorm()
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): LMHead(
    (proj): Linear(in_features=960, out_features=960, bias=True)
    (act): GELU(approximate='none')
    (ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
    (proj_final): Linear(in_features=960, out_features=30, bias=True)
  )
)

In [7]:
l = [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 13,  1,  9,  1,  1,
         1,  1,  1,  1,  1,  1,  1, 22,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1, 19,  1,  1,  1,  1,  1,  1,  1,  1, 19,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 16,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1]

l.count(1) / len(l)

0.9405940594059405